In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
"""
LSTM-based residual predictor for multi-object tracking.

Architecture
------------
Input (per track, per frame) — 10-dim normalised feature:
    [cx/W, cy/H, w/W, h/H, vx/W, vy/H, delta_t, is_missing,
     missing_count/max_missing, confidence]

LSTM (stacked, batch_first) processes a *single timestep* per
forward call during inference so that each track maintains its
own (h, c) state between frames.

Output
------
    residual  [4]  — bbox correction [dcx, dcy, dw, dh] in pixel space,
                     added on top of Kalman prediction.
    h_new, c_new  — updated LSTM hidden and cell states.

Zero-initialisation on the residual head ensures the model begins
as a pure Kalman filter before any training.
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class LSTMPredictor(nn.Module):
    INPUT_DIM = 10

    def __init__(
        self,
        hidden_size: int = 128,
        num_layers: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Input projection (normalises scale before LSTM)
        self.input_proj = nn.Linear(self.INPUT_DIM, hidden_size)

        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.residual_head = nn.Linear(hidden_size, 4)

        # Zero-init residual → pure-Kalman behaviour before training
        nn.init.zeros_(self.residual_head.weight)
        nn.init.zeros_(self.residual_head.bias)

    # ------------------------------------------------------------------
    # Single-step inference  (used by the tracker)
    # ------------------------------------------------------------------

    def step(
        self,
        x: torch.Tensor,      # [B, INPUT_DIM]
        h: torch.Tensor,      # [num_layers, B, hidden_size]
        c: torch.Tensor,      # [num_layers, B, hidden_size]
    ):
        """One-step forward, returns residual, h_new, c_new."""
        h = h.contiguous()
        c = c.contiguous()
        feat = F.relu(self.input_proj(x))          # [B, H]
        out, (h_new, c_new) = self.lstm(
            feat.unsqueeze(1), (h, c)              # seq_len=1
        )
        out = out.squeeze(1)                        # [B, H]
        return self.residual_head(out), h_new, c_new

    # ------------------------------------------------------------------
    # Sequence forward  (used during training)
    # ------------------------------------------------------------------

    def forward(
        self,
        x_seq: torch.Tensor,   # [B, T, INPUT_DIM]
        h0: torch.Tensor,      # [num_layers, B, H]
        c0: torch.Tensor,      # [num_layers, B, H]
    ):
        """Full-sequence forward for training (BPTT)."""
        feat = F.relu(self.input_proj(x_seq))      # [B, T, H]
        out, (h_n, c_n) = self.lstm(feat, (h0, c0))
        residuals = self.residual_head(out)         # [B, T, 4]
        return residuals, h_n, c_n

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    def init_hidden(
        self,
        batch_size: int = 1,
        device: torch.device | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if device is None:
            device = next(self.parameters()).device
        h = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)
        c = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)
        return h, c

    @staticmethod
    def build_input(
        bbox_cxcywh: np.ndarray,   # [cx, cy, w, h] pixel space
        velocity: np.ndarray,       # [vx, vy]  pixels / frame
        delta_t: float,             # frames elapsed since last update
        is_missing: int,            # 1 if no detection this frame
        missing_count: int,         # consecutive missing frames so far
        confidence: float,          # detection score (0 when missing)
        img_w: float,
        img_h: float,
        max_missing: float = 30.0,
    ) -> np.ndarray:               # float32  [INPUT_DIM]
        cx, cy, w, h = bbox_cxcywh
        vx, vy = velocity
        return np.array([
            cx / img_w,
            cy / img_h,
            w / img_w,
            h / img_h,
            vx / img_w,
            vy / img_h,
            float(delta_t),
            float(is_missing),
            min(missing_count / max_missing, 1.0),
            float(confidence),
        ], dtype=np.float32)

    @staticmethod
    def huber_loss(
        residuals: torch.Tensor,  # [B, T, 4]  predicted
        targets: torch.Tensor,    # [B, T, 4]  GT residuals
        mask: torch.Tensor,       # [B, T]     1 = valid frame, 0 = padding
        delta: float = 1.0,
    ) -> torch.Tensor:
        """
        Huber loss masked over valid frames.
        More robust than MSE: linear for large errors, quadratic for small ones.
        """
        per_dim = F.huber_loss(residuals, targets, reduction="none", delta=delta)
        loss = per_dim.mean(dim=-1)                 # [B, T]
        loss = (loss * mask).sum() / mask.sum().clamp(min=1)
        return loss


In [ ]:
"""
MOTTrackletDataset — builds LSTM training sequences from COCO-format
annotations (the same format ByteTrack already uses).

Data flow
─────────
1. Load COCO JSON (mot/annotations/train.json or mix_det/annotations/train.json).
2. Group annotations by (video_id, track_id) → raw tracklets.
3. For each tracklet:
   a. Simulate Kalman filter frame-by-frame on the GT bboxes.
   b. Compute target residual = GT_cxcywh − KF_predicted_cxcywh.
   c. Randomly drop frames (simulate occlusion); set is_missing=1 for those.
   d. Slide a window of length `seq_len` over the tracklet.
4. Each sample: (x_seq [T, 10], target_residual [T, 4], mask [T]).

The mask is 1 for non-missing frames (where we compute the loss),
0 for artificially dropped frames.
"""

import json
import os
import random
import numpy as np
from torch.utils.data import Dataset

from yolox.tracker.kalman_filter import KalmanFilter


# ── Kalman helpers ────────────────────────────────────────────────────────────

def _tlwh_to_xyah(tlwh: np.ndarray) -> np.ndarray:
    """[x1, y1, w, h] → [cx, cy, aspect, h]"""
    ret = tlwh.copy().astype(np.float64)
    ret[0] += ret[2] / 2
    ret[1] += ret[3] / 2
    ret[2] = ret[2] / ret[3] if ret[3] > 0 else 1.0
    return ret


def _xyah_to_cxcywh(xyah: np.ndarray) -> np.ndarray:
    """[cx, cy, aspect, h] → [cx, cy, w, h]"""
    cx, cy, a, h = xyah
    return np.array([cx, cy, a * h, h], dtype=np.float32)


def simulate_kalman(
    tlwh_seq: np.ndarray,        # [T, 4]  GT bboxes (tlwh)
    frame_ids: np.ndarray,       # [T]     frame indices (may have gaps)
    miss_flags: np.ndarray,      # [T]     1 = artificially dropped
) -> tuple[np.ndarray, np.ndarray]:
    """
    Run a Kalman filter over a tracklet, skipping dropped frames.

    Returns
    -------
    kf_preds  : [T, 4]  KF-predicted bbox in [cx, cy, w, h] (before update)
    residuals : [T, 4]  GT_cxcywh − KF_pred (0 for missing frames)
    """
    kf = KalmanFilter()
    T = len(tlwh_seq)
    kf_preds = np.zeros((T, 4), dtype=np.float32)
    residuals = np.zeros((T, 4), dtype=np.float32)

    mean, cov = None, None

    for i, (tlwh, fid, is_miss) in enumerate(zip(tlwh_seq, frame_ids, miss_flags)):
        gt_cxcywh = np.array(
            [tlwh[0] + tlwh[2] / 2, tlwh[1] + tlwh[3] / 2, tlwh[2], tlwh[3]],
            dtype=np.float32,
        )
        xyah = _tlwh_to_xyah(tlwh)

        if mean is None:
            # First frame: initialise KF
            mean, cov = kf.initiate(xyah)
            kf_preds[i] = gt_cxcywh   # no prediction yet → no residual
            residuals[i] = 0.0
        else:
            # Predict step
            mean, cov = kf.predict(mean, cov)
            pred_cxcywh = _xyah_to_cxcywh(mean[:4])
            kf_preds[i] = pred_cxcywh

            if is_miss == 0 and tlwh[2] > 0 and tlwh[3] > 0:
                residuals[i] = gt_cxcywh - pred_cxcywh
                mean, cov = kf.update(mean, cov, xyah)
            # else: miss or degenerate bbox → skip KF update, residual stays 0

    return kf_preds, residuals


# ── Dataset ───────────────────────────────────────────────────────────────────

class MOTTrackletDataset(Dataset):
    """
    Parameters
    ----------
    ann_files    : list of COCO-format JSON paths (can mix datasets).
    seq_len      : sliding-window length (frames).
    miss_prob    : probability of randomly dropping a detected frame.
    min_track_len: discard tracklets shorter than this.
    max_missing  : normalisation denominator for missing_count feature.
    augment      : if True, apply random horizontal flip to bboxes.
    """

    def __init__(
        self,
        ann_files: str | os.PathLike | list[str | os.PathLike],
        seq_len: int = 32,
        miss_prob: float = 0.15,
        min_track_len: int = 8,
        max_missing: float = 30.0,
        augment: bool = True,
    ):
        self.seq_len = seq_len
        self.miss_prob = miss_prob
        self.max_missing = max_missing
        self.augment = augment

        self.samples: list[dict] = []   # each sample = one sliding window
        if isinstance(ann_files, (str, os.PathLike)):
            ann_files = [ann_files]
        self._build(ann_files, min_track_len)

    # ── Build ─────────────────────────────────────────────────────────────

    def _build(self, ann_files: list[str | os.PathLike], min_track_len: int):
        for ann_file in ann_files:
            print(f"[Dataset] loading {ann_file}")
            with open(ann_file) as f:
                data = json.load(f)

            # image_id → metadata
            img_meta: dict[int, dict] = {
                img["id"]: img for img in data["images"]
            }

            # group annotations by (video_id, track_id)
            tracklets: dict[tuple, list] = {}
            for ann in data["annotations"]:
                img = img_meta[ann["image_id"]]
                key = (img["video_id"], ann["track_id"])
                tracklets.setdefault(key, []).append({
                    "frame_id": img["frame_id"],
                    "bbox": ann["bbox"],          # [x, y, w, h]
                    "conf": ann.get("conf", 1.0),
                    "img_w": img["width"],
                    "img_h": img["height"],
                })

            for key, frames in tracklets.items():
                frames.sort(key=lambda x: x["frame_id"])
                if len(frames) < min_track_len:
                    continue
                self._add_tracklet(frames)

        print(f"[Dataset] total samples: {len(self.samples)}")

    def _add_tracklet(self, frames: list[dict]):
        """Slide a window over one tracklet and store samples."""
        # Drop frames with zero/negative width or height (degenerate annotations)
        frames = [f for f in frames if f["bbox"][2] > 0 and f["bbox"][3] > 0]
        if len(frames) < 2:
            return

        T_full = len(frames)
        img_w = float(frames[0]["img_w"])
        img_h = float(frames[0]["img_h"])

        tlwh_arr = np.array([f["bbox"] for f in frames], dtype=np.float32)
        fid_arr = np.array([f["frame_id"] for f in frames], dtype=np.int32)
        conf_arr = np.array([f["conf"] for f in frames], dtype=np.float32)

        stride = max(1, self.seq_len // 4)
        starts = list(range(0, T_full - self.seq_len + 1, stride))
        if not starts:
            starts = [0]

        for start in starts:
            end = min(start + self.seq_len, T_full)
            self.samples.append({
                "tlwh": tlwh_arr[start:end],
                "fid": fid_arr[start:end],
                "conf": conf_arr[start:end],
                "img_w": img_w,
                "img_h": img_h,
            })

    # ── Item ──────────────────────────────────────────────────────────────

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        s = self.samples[idx]
        tlwh = s["tlwh"].copy()         # [T, 4]
        fid = s["fid"].copy()
        conf = s["conf"].copy()
        img_w, img_h = s["img_w"], s["img_h"]
        T = len(tlwh)

        # Pad if shorter than seq_len
        if T < self.seq_len:
            pad = self.seq_len - T
            tlwh = np.pad(tlwh, ((0, pad), (0, 0)))
            fid = np.pad(fid, (0, pad), constant_values=fid[-1] + 1)
            conf = np.pad(conf, (0, pad))

        # Optional horizontal flip augmentation
        if self.augment and random.random() < 0.5:
            tlwh[:, 0] = img_w - tlwh[:, 0] - tlwh[:, 2]

        # Randomly drop some frames (occlusion simulation)
        miss_flags = np.zeros(self.seq_len, dtype=np.float32)
        for i in range(1, self.seq_len):    # never drop the first frame
            if random.random() < self.miss_prob:
                miss_flags[i] = 1.0
        # Padded frames must be treated as missing so KF never receives
        # zero-bbox measurements (which would break positive-definiteness)
        miss_flags[T:] = 1.0

        # Valid mask: 1 on observed, non-padded frames
        valid_mask = np.ones(self.seq_len, dtype=np.float32)
        valid_mask[T:] = 0.0
        valid_mask[miss_flags == 1] = 0.0  # don't supervise missing frames

        # Simulate Kalman → residuals
        kf_preds, residuals = simulate_kalman(
            tlwh[:self.seq_len], fid[:self.seq_len], miss_flags
        )

        # Build LSTM input features
        cxcywh = np.stack([
            tlwh[:, 0] + tlwh[:, 2] / 2,
            tlwh[:, 1] + tlwh[:, 3] / 2,
            tlwh[:, 2],
            tlwh[:, 3],
        ], axis=-1).astype(np.float32)   # [T, 4]

        x_seq = self._build_feature_seq(
            cxcywh, miss_flags, conf, fid[:self.seq_len], img_w, img_h
        )  # [T, 10]

        # Residuals kept in pixel space — LSTM outputs pixels, applied directly
        # to KF bbox at inference without any denorm step.

        return {
            "x_seq": x_seq.astype(np.float32),           # [T, 10]
            "target_residual": residuals.astype(np.float32),  # [T, 4]
            "mask": valid_mask.astype(np.float32),        # [T]
        }

    # ── Feature construction ──────────────────────────────────────────────

    @staticmethod
    def _build_feature_seq(
        cxcywh: np.ndarray,      # [T, 4]
        miss_flags: np.ndarray,  # [T]
        conf: np.ndarray,        # [T]
        fid: np.ndarray,         # [T]  actual frame indices
        img_w: float,
        img_h: float,
        max_missing: float = 30.0,
    ) -> np.ndarray:             # [T, 10]
        T = len(cxcywh)
        x_seq = np.zeros((T, 10), dtype=np.float32)
        missing_count = 0

        for i in range(T):
            cx, cy, w, h = cxcywh[i]
            if i == 0:
                vx, vy = 0.0, 0.0
                delta_t = 1.0
            else:
                dt = float(fid[i] - fid[i - 1])
                delta_t = max(dt, 1.0)   # guard against padding artifacts
                vx = (cx - cxcywh[i - 1, 0]) / delta_t
                vy = (cy - cxcywh[i - 1, 1]) / delta_t

            is_miss = float(miss_flags[i])
            missing_count = missing_count + 1 if is_miss else 0
            score = 0.0 if is_miss else float(conf[i])

            x_seq[i] = [
                cx / img_w,
                cy / img_h,
                w / img_w,
                h / img_h,
                vx / img_w,
                vy / img_h,
                delta_t,
                is_miss,
                min(missing_count / max_missing, 1.0),
                score,
            ]
        return x_seq


In [ ]:
"""
Notebook-friendly LSTM MOT training API.

Usage:
    cfg = LSTMMOTTrainConfig(
        ann_files=["datasets/mot/annotations/train.json"],
        val_ann_files=["datasets/mot/annotations/val_half.json"],
        save_dir="checkpoints/lstm",
        epochs=30,
    )
    result = train_lstm_mot(cfg)

Or pass parameters directly:
    result = train_lstm_mot(
        ann_files=["datasets/mot/annotations/train.json"],
        val_ann_files=["datasets/mot/annotations/val_half.json"],
        save_dir="checkpoints/lstm",
        epochs=30,
    )
"""

import os
import random
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

try:
    _PROJECT_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    _PROJECT_ROOT = Path.cwd()

if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

try:
    LSTMPredictor
except NameError:
    from new_pipeline.lstm_predictor import LSTMPredictor

try:
    MOTTrackletDataset
except NameError:
    from new_pipeline.dataset import MOTTrackletDataset


@dataclass
class LSTMMOTTrainConfig:
    ann_files: str | os.PathLike | Sequence[str | os.PathLike]
    val_ann_files: Optional[str | os.PathLike | Sequence[str | os.PathLike]] = None
    seq_len: int = 32
    miss_prob: float = 0.15
    val_ratio: float = 0.05
    hidden_size: int = 128
    num_layers: int = 2
    dropout: float = 0.1
    epochs: int = 30
    save_every: int = 5
    batch_size: int = 64
    lr: float = 1e-3
    weight_decay: float = 1e-4
    grad_clip: float = 5.0
    warmup_epochs: int = 2
    patience: int = 8
    save_dir: str | os.PathLike = "checkpoints/lstm"
    resume: Optional[str | os.PathLike] = None
    device: Optional[str] = None
    num_workers: int = 4
    seed: int = 42

    def __post_init__(self):
        self.ann_files = _as_path_list(self.ann_files)
        self.val_ann_files = None if self.val_ann_files is None else _as_path_list(self.val_ann_files)
        self.save_dir = str(self.save_dir)
        self.resume = None if self.resume is None else str(self.resume)
        if not self.ann_files:
            raise ValueError("ann_files must contain at least one annotation file")
        if self.epochs <= 0:
            raise ValueError("epochs must be > 0")
        if self.batch_size <= 0:
            raise ValueError("batch_size must be > 0")
        if self.seq_len <= 0:
            raise ValueError("seq_len must be > 0")
        if not 0.0 <= self.val_ratio < 1.0:
            raise ValueError("val_ratio must be in [0, 1)")


def _as_path_list(value) -> list[str]:
    if isinstance(value, (str, os.PathLike)):
        return [str(value)]
    return [str(v) for v in value]


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def warmup_lambda(epoch: int, warmup_epochs: int) -> float:
    if warmup_epochs <= 0:
        return 1.0
    if epoch < warmup_epochs:
        return float(epoch + 1) / float(warmup_epochs)
    return 1.0


def save_checkpoint(model: LSTMPredictor, optimizer, scheduler, epoch: int,
                    val_loss: float, path: str):
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "val_loss": val_loss,
    }, path)


def run_epoch(model: LSTMPredictor, loader: DataLoader, optimizer, device: torch.device,
              grad_clip: float, training: bool, desc: str = "") -> float:
    model.train(training)
    total_loss = 0.0
    total_samples = 0

    pbar = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True)
    for batch in pbar:
        x_seq = batch["x_seq"].to(device)
        target = batch["target_residual"].to(device)
        mask = batch["mask"].to(device)
        batch_size = x_seq.size(0)

        h0 = torch.zeros(model.num_layers, batch_size, model.hidden_size, device=device)
        c0 = torch.zeros(model.num_layers, batch_size, model.hidden_size, device=device)

        with torch.set_grad_enabled(training):
            residuals, _, _ = model(x_seq, h0, c0)
            loss = LSTMPredictor.huber_loss(residuals, target, mask)

        if training:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        total_loss += loss.item() * batch_size
        total_samples += batch_size
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(total_samples, 1)


def train_lstm_mot(config: Optional[LSTMMOTTrainConfig] = None, **kwargs):
    if config is None:
        config = LSTMMOTTrainConfig(**kwargs)
    elif kwargs:
        raise ValueError("Pass either config or keyword arguments, not both")

    set_seed(config.seed)
    device = torch.device(
        config.device if config.device else ("cuda" if torch.cuda.is_available() else "cpu")
    )
    print(f"[train] device: {device}")

    os.makedirs(config.save_dir, exist_ok=True)

    train_ds = MOTTrackletDataset(
        ann_files=config.ann_files,
        seq_len=config.seq_len,
        miss_prob=config.miss_prob,
    )

    if config.val_ann_files:
        val_ds = MOTTrackletDataset(
            ann_files=config.val_ann_files,
            seq_len=config.seq_len,
            miss_prob=0.0,
            augment=False,
        )
    else:
        n_val = max(1, int(len(train_ds) * config.val_ratio))
        n_train = len(train_ds) - n_val
        generator = torch.Generator().manual_seed(config.seed)
        train_ds, val_ds = random_split(train_ds, [n_train, n_val], generator=generator)

    print(f"[train] train={len(train_ds)} val={len(val_ds)}")

    train_loader = DataLoader(
        train_ds,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=device.type == "cuda",
        drop_last=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=device.type == "cuda",
    )

    model = LSTMPredictor(
        hidden_size=config.hidden_size,
        num_layers=config.num_layers,
        dropout=config.dropout,
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config.epochs, eta_min=config.lr * 0.01
    )
    warmup = optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda e: warmup_lambda(e, config.warmup_epochs)
    )

    start_epoch = 0
    best_val = float("inf")
    if config.resume:
        ckpt = torch.load(config.resume, map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch = ckpt["epoch"] + 1
        best_val = ckpt.get("val_loss", float("inf"))
        print(f"[train] resumed from epoch {start_epoch}, best_val={best_val:.6f}")

    patience_counter = 0
    epoch_bar = tqdm(range(start_epoch, config.epochs), desc="epochs", dynamic_ncols=True)
    for epoch in epoch_bar:
        lr = optimizer.param_groups[0]["lr"]
        train_loss = run_epoch(
            model, train_loader, optimizer, device, config.grad_clip,
            training=True, desc=f"train e{epoch + 1}",
        )
        val_loss = run_epoch(
            model, val_loader, optimizer, device, config.grad_clip,
            training=False, desc=f"val e{epoch + 1}",
        )

        if epoch < config.warmup_epochs:
            warmup.step()
        else:
            scheduler.step()

        improved = val_loss < best_val
        flag = " <- best" if improved else ""
        epoch_bar.write(
            f"[epoch {epoch + 1:03d}/{config.epochs}]"
            f" train={train_loss:.6f} val={val_loss:.6f}"
            f" lr={lr:.2e}{flag}"
        )

        if (epoch + 1) % config.save_every == 0:
            save_checkpoint(
                model, optimizer, scheduler, epoch, val_loss,
                os.path.join(config.save_dir, f"epoch_{epoch + 1:03d}.pth"),
            )

        if improved:
            best_val = val_loss
            patience_counter = 0
            save_checkpoint(
                model, optimizer, scheduler, epoch, val_loss,
                os.path.join(config.save_dir, "best.pth"),
            )
        else:
            patience_counter += 1
            if patience_counter >= config.patience:
                epoch_bar.write(f"[train] early stopping at epoch {epoch + 1}")
                break

    best_checkpoint = os.path.join(config.save_dir, "best.pth")
    print(f"[train] done. best val loss: {best_val:.6f}")
    print(f"[train] checkpoint: {best_checkpoint}")

    return {
        "model": model,
        "config": config,
        "best_val": best_val,
        "best_checkpoint": best_checkpoint,
    }
